# 08. Tópicos en el Tiempo 

Este cuaderno evalúa la evolución temporal de la agenda editorial de Revista Anfibia (2012–presente) a partir del modelo BERTopic entrenado.

**Objetivos:**
1. Integrar la dimensión temporal (`post_date`) sin necesidad de re-entrenar el modelo de lenguaje base.
2. Calcular la dinámica de tópicos mediante `topics_over_time` (evolución del c-TF-IDF y volumen de notas).
3. Medir la estacionalidad para clasificar tópicos entre **coyunturales** (event-driven / picos) y **estructurales** (cobertura continua / evergreen).

*El modelo usado es el hibrido, pero se puede cambiar por la variables NOMBRE_MODELO y CARPETA_MODELO en la celda siguiente*

In [6]:
# 1. Parche de compatibilidad obligatorio (Numba/Coverage)
import coverage.types as ct
for name, fallback in {
    'Tracer': getattr(ct, 'TTracer', object),
    'TShouldTraceFn': object,
    'TShouldStartContextFn': object,
    'TFileDisposition': object,
}.items():
    if not hasattr(ct, name):
        setattr(ct, name, fallback)

# 2. Imports generales
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from bertopic import BERTopic

warnings.filterwarnings("ignore")

# 3. Configuración de Rutas y Variables
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# VARIABLE CONFIGURABLE DE MODELO
CARPETA_MODELO = "bertopic_hibrido"
NOMBRE_MODELO = "bertopic_hibrido_model"

MODEL_PATH = PROJECT_ROOT / "outputs" / CARPETA_MODELO / NOMBRE_MODELO
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed" / "corpus_anfibia_lematizado.csv"
RAW_POSTS = PROJECT_ROOT / "data" / "processed" / "posteos_anfibia.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "analisis_temporal"
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

# 4. Carga de datos y modelo
print(f"Cargando modelo: {MODEL_PATH}")
topic_model = BERTopic.load(MODEL_PATH)

df = pd.read_csv(DATA_PROCESSED)

# Si post_date no está en el corpus lematizado, lo traemos del raw sin pisar duplicados
if "post_date" not in df.columns:
    df_fechas = pd.read_csv(RAW_POSTS, usecols=["ID", "post_date", "post_year"])
    # Nos aseguramos de traer solo lo que falte
    cols_a_traer = [c for c in ["ID", "post_date", "post_year"] if c not in df.columns or c == "ID"]
    df = df.merge(df_fechas[cols_a_traer], on="ID", how="left")

df["post_date"] = pd.to_datetime(df["post_date"], errors="coerce")

# Asignar tópicos (usa los existentes si ya están guardados para ahorrar tiempo)
if "topic" not in df.columns:
    topics, _ = topic_model.transform(df["document_text_final"].fillna("").tolist())
    df["topic"] = topics

# Filtrar registros válidos con fecha y texto
df_temp = df[df["post_date"].notna() & df["document_text_final"].notna()].sort_values("post_date").reset_index(drop=True)

print(f"Total de artículos listos para análisis temporal: {len(df_temp):,}")
print(f"Rango de fechas: {df_temp['post_date'].min().strftime('%Y-%m-%d')} a {df_temp['post_date'].max().strftime('%Y-%m-%d')}")

Cargando modelo: C:\Trabajos\anfibia-unsam\outputs\bertopic_hibrido\bertopic_hibrido_model
Total de artículos listos para análisis temporal: 3,556
Rango de fechas: 2012-05-14 a 2026-06-16


## 1. Analisis topicos en el tiempo
Calculamos la evolución de frecuencias y el desplazamiento de términos representativos en el tiempo.

In [7]:
timestamps = df_temp["post_date"].dt.to_pydatetime().tolist()
docs = df_temp["document_text_final"].tolist()

print("Calculando evolución temporal de tópicos (Topics over Time)...")
topics_over_time = topic_model.topics_over_time(
    docs=docs,
    timestamps=timestamps,
    global_tuning=True,
    evolution_tuning=True,
    nr_bins=25  # Ventanas de tiempo a lo largo del histórico
)

# Exportar tabla cruda de evolución
topics_over_time.to_csv(OUTPUT_DIR / "topics_over_time_raw.csv", index=False)
display(topics_over_time.head(10))

Calculando evolución temporal de tópicos (Topics over Time)...


,Topic,Words,Frequency,Timestamp
0,-1,"carlos, kaiser, padre, junto, velázquez",7,2012-05-09 10:13:18.781
1,0,"diario, lorenzo, hombre, contar, mundo",17,2012-05-09 10:13:18.781
2,9,"pescador, barco, marino, marinero, marisco",1,2012-05-09 10:13:18.781
3,10,"rebelión, manifestación, pueblo, marcha, fiesta",1,2012-05-09 10:13:18.781
4,-1,"hombre, junto, domador, caballo, contar",8,2012-12-06 09:45:30.760
5,0,"muerto, hombre, contar, junto, sino",9,2012-12-06 09:45:30.760
6,3,"pesticida, humo, pandemia, tóxico, peligroso",1,2012-12-06 09:45:30.760
7,7,"israel, israelí, palestina, palestino, aviv",1,2012-12-06 09:45:30.760
8,-1,"periodista, obra, contar, sino, pensar",18,2013-06-30 05:47:39.520
9,0,"político, política, candidato, tal, padre",11,2013-06-30 05:47:39.520


## 2. Métricas de Estacionalidad y Clasificación 

Para entender el comportamiento de cada tópico en el tiempo calculamos:
* **Volumen Total:** Cantidad histórica de artículos.
* **Pico Máximo:** Mayor concentración de notas en una sola ventana temporal.
* **Coeficiente de Variación ($CV = \frac{\sigma}{\mu}$):** 
  * $CV$ **alto:** Tópico **coyuntural / event-driven** (picos concentrados en momentos específicos, ej. elecciones, leyes, pandemia).
  * $CV$ **bajo:** Tópico **estructural / evergreen** (producción editorial constante a lo largo de toda la década).

In [8]:
# 1. Filtramos el ruido (-1) para el análisis editorial
tot_validos = topics_over_time[topics_over_time["Topic"] != -1].copy()

# 2. Agrupación y cálculo estadístico por tópico
metricas_estabilidad = (
    tot_validos.groupby("Topic")
    .agg(
        volumen_total=("Frequency", "sum"),
        pico_maximo=("Frequency", "max"),
        frecuencia_media=("Frequency", "mean"),
        desvio_std=("Frequency", "std"),
        terminos_historicos=("Words", "first")
    )
    .reset_index()
)

# 3. Coeficiente de variación (llenamos con 0 si desvío es NaN por baja frecuencia)
metricas_estabilidad["desvio_std"] = metricas_estabilidad["desvio_std"].fillna(0)
metricas_estabilidad["coef_variacion"] = (
    metricas_estabilidad["desvio_std"] / metricas_estabilidad["frecuencia_media"]
).round(2)

# 4. Clasificación: usamos el percentil 60 del CV como corte empírico
umbral_coyuntural = metricas_estabilidad["coef_variacion"].quantile(0.60)

metricas_estabilidad["tipo_agenda"] = np.where(
    metricas_estabilidad["coef_variacion"] >= umbral_coyuntural,
    "Coyuntural / Event-driven",
    "Estructural / Evergreen"
)

# 5. Ordenamos por los más coyunturales (mayor CV)
metricas_estabilidad = metricas_estabilidad.sort_values(by="coef_variacion", ascending=False).reset_index(drop=True)

# Guardar métricas procesadas
metricas_estabilidad.to_csv(OUTPUT_DIR / "clasificacion_temporal_topicos.csv", index=False)

print(f"Umbral de corte de CV (Percentil 60): {umbral_coyuntural:.2f}")
display(metricas_estabilidad.head(15))

Umbral de corte de CV (Percentil 60): 0.61


,Topic,volumen_total,pico_maximo,frecuencia_media,desvio_std,terminos_historicos,coef_variacion,tipo_agenda
0,2,86,35,5.058824,8.613003,"abortar, aborto, política, legislador, debate",1.70,Coyuntural / Event-driven
1,3,75,21,5.000000,6.244998,"pesticida, humo, pandemia, tóxico, peligroso",1.25,Coyuntural / Event-driven
2,4,46,9,2.421053,2.168353,"foucault, sorbonne, autor, coloquio, françois",0.90,Coyuntural / Event-driven
3,1,113,19,5.380952,4.421269,"facebook, investigador, zuckerberg, internet, ...",0.82,Coyuntural / Event-driven
4,11,13,6,3.250000,2.217356,"terrorista, terrorismo, europeo, musulmán, isl...",0.68,Coyuntural / Event-driven
5,6,27,4,1.588235,0.939336,"soviético, vladimir, rusia, kremlin, ruso",0.59,Estructural / Evergreen
6,8,16,3,1.142857,0.534522,"biólogo, amnesia, cerebro, memoria, científico",0.47,Estructural / Evergreen
7,7,19,3,1.727273,0.786245,"israel, israelí, palestina, palestino, aviv",0.46,Estructural / Evergreen
8,5,30,3,1.764706,0.752447,"artista, estética, arte, artístico, coleccionista",0.43,Estructural / Evergreen
9,0,1686,105,67.440000,26.124829,"diario, lorenzo, hombre, contar, mundo",0.39,Estructural / Evergreen


Teniendolo en cuenta para la recomendacion de articulos, los artículos de tópicos coyunturales (CV > 0.65) podrian recibir una penalización por antigüedad (*time decay*), mientras que los tópicos estructurales (CV < 0.50) pueden recomendar notas históricas sin perder vigencia.

## 2.5 Inspección Cualitativa de Tópicos
Analizemos estos topicos clasificados. Mapeamos las palabras clave algorítmicas con los títulos reales de las notas.

In [17]:
print("INSPECCION EDITORIAL DE TOPICOS PRINCIPALES")
print("="*80)

# Filtramos ruido (-1) y el macro-topico(politica) (0)
topicos_limpios = metricas_estabilidad[
    (metricas_estabilidad["Topic"] != -1) & 
    (metricas_estabilidad["Topic"] != 0)
]

# Agarramos los topicos con al menos 20 notas y seleccionamos los 8 de mayor volumen
notas = 20
seleccion = 8
topicos_a_revisar = topicos_limpios[topicos_limpios["volumen_total"] >= notas] \
                    .sort_values(by="volumen_total", ascending=False) \
                    .head(seleccion)["Topic"].tolist()

for t in topicos_a_revisar:
    # Traemos la metadata del topico
    datos_topico = metricas_estabilidad[metricas_estabilidad["Topic"] == t].iloc[0]
    notas_topico = df_temp[df_temp["topic"] == t]
    
    print(f"\nTOPICO {t} | Tipo: {datos_topico['tipo_agenda']}")
    print(f"Palabras clave: {datos_topico['terminos_historicos']}")
    print(f"Volumen total: {len(notas_topico)} notas")
    print("Ejemplos de articulos:")
    
    # Seleccionamos 5 notas al azar
    ejemplos = notas_topico.sample(min(5, len(notas_topico)), random_state=42)
    
    for _, row in ejemplos.iterrows():
        # Limpiamos saltos de linea en los titulos
        titulo = str(row['title_text']).replace('\n', ' ').strip()
        # Extraemos el año directamente del datetime
        anio = row['post_date'].year
        print(f"   - [{anio}] {titulo}")

print("\n" + "="*80)

INSPECCION EDITORIAL DE TOPICOS PRINCIPALES

TOPICO 1 | Tipo: Coyuntural / Event-driven
Palabras clave: facebook, investigador, zuckerberg, internet, científico
Volumen total: 88 notas
Ejemplos de articulos:
   - [2024] El mapa de medios en la era Milei
   - [2014] Nosotros, los cobayos de Facebook
   - [2017] Te amo, te odio, dame más
   - [2017] Gente que sí
   - [2016] Tecno optimismo

TOPICO 2 | Tipo: Coyuntural / Event-driven
Palabras clave: abortar, aborto, política, legislador, debate
Volumen total: 86 notas
Ejemplos de articulos:
   - [2021] ¿Se puede ser racista y xenófobx y apoyar el aborto legal?
   - [2014] La caída de la súper mujer
   - [2020] "Yo salí para que salga"
   - [2018] La doble moral médica
   - [2018] El aborto es radical

TOPICO 3 | Tipo: Coyuntural / Event-driven
Palabras clave: pesticida, humo, pandemia, tóxico, peligroso
Volumen total: 63 notas
Ejemplos de articulos:
   - [2023] ¿Quién puede ver una peli de un tirón?
   - [2021] Un país que pone el brazo
 

## 3. Visualización

Generamos dos visualizaciones interactivas:
1. **Curvas Globales:** Trayectoria de los tópicos principales a lo largo de toda la historia.
2. **Coyunturales vs. Estructurales:** Comparativa directa temas que tuvieron un pico en un momento (ej. Aborto, Pandemia) frente a temas frecuentes (ej. Arte, Ciencia).

In [ ]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# ==============================================================================
# 1. PREPARACIÓN ESTADÍSTICA ESTRICTA
# ==============================================================================

# Filtramos Tópico -1 (ruido) y Tópico 0 (el megaclúster que aplasta al resto)
topicos_limpios = metricas_estabilidad[
    (metricas_estabilidad["Topic"] != -1) & 
    (metricas_estabilidad["Topic"] != 0)
].copy()

# Regla 1: Solo tópicos con volumen decente (> 25 notas) para que la curva tenga sentido
topicos_robustos = topicos_limpios[topicos_limpios["volumen_total"] >= 25].copy()

# Preparar dataset temporal
df_evo = df_temp[df_temp["topic"].isin(topicos_robustos["Topic"])].copy()
df_evo["post_year"] = df_evo["post_date"].dt.year
agg = df_evo.groupby(["topic", "post_year"]).size().reset_index(name="notas")

# Regla 2: Normalización por Pico Máximo (El pico histórico = 100%)
max_por_topico = agg.groupby("topic")["notas"].max().reset_index(name="max_notas")
agg = agg.merge(max_por_topico, on="topic")
agg["intensidad_relativa"] = (agg["notas"] / agg["max_notas"]) * 100

# Separar extremos reales
top_coyuntura = topicos_robustos.sort_values("coef_variacion", ascending=False).head(3)
top_evergreen = topicos_robustos.sort_values("coef_variacion", ascending=True).head(3)

# ==============================================================================
# GRÁFICO A: EL CUADRANTE EDITORIAL (Scatter Analítico)
# ==============================================================================
# Este gráfico ubica a todos los tópicos robustos en un cuadrante: Volumen vs Estacionalidad
fig_scatter = px.scatter(
    topicos_robustos,
    x="volumen_total",
    y="coef_variacion",
    color="tipo_agenda",
    size="volumen_total",
    hover_name="terminos_historicos",
    text="Topic",
    color_discrete_map={"Coyuntural / Event-driven": "#d62728", "Estructural / Evergreen": "#2ca02c"},
    title="<b>Cuadrante Editorial: Coyuntura vs. Estructura (Tópicos > 25 notas)</b>",
    labels={"volumen_total": "Volumen Total de Notas", "coef_variacion": "Nivel de Coyuntura (Coef. Variación)"},
    height=500
)
fig_scatter.update_traces(textposition='top center')
fig_scatter.add_hline(y=umbral_coyuntural, line_dash="dash", line_color="gray", annotation_text="Umbral de Coyuntura")
fig_scatter.write_html(OUTPUT_DIR / "03A_cuadrante_editorial.html")
fig_scatter.show()

# ==============================================================================
# GRÁFICO B: CURVAS DE INTENSIDAD
# ==============================================================================
fig_contrast = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        "<b>🔴 PICOS DE COYUNTURA</b><br><sup>Llegan al 100% por un evento y caen</sup>",
        "<b>🟢 FLUJO EVERGREEN</b><br><sup>Oscilan sin apagarse nunca (Base constante)</sup>"
    ),
    shared_yaxes=True
)

colores_coyuntura = ["#d62728", "#ff7f0e", "#9467bd"]
colores_evergreen = ["#2ca02c", "#1f77b4", "#17becf"]

# Trazar Coyuntura
for idx, (_, row) in enumerate(top_coyuntura.iterrows()):
    t_id = row["Topic"]
    sub = agg[agg["topic"] == t_id].sort_values("post_year")
    label = f"T{t_id}: {row['terminos_historicos'][:25]}..."
    
    fig_contrast.add_trace(
        go.Scatter(
            x=sub["post_year"], y=sub["intensidad_relativa"],
            mode="lines+markers", name=label,
            line=dict(width=3, color=colores_coyuntura[idx % len(colores_coyuntura)]),
            hovertemplate=f"<b>{label}</b><br>Año: %{{x}}<br>Intensidad: %{{y:.0f}}% de su pico<extra></extra>"
        ), row=1, col=1
    )

# Trazar Evergreen
for idx, (_, row) in enumerate(top_evergreen.iterrows()):
    t_id = row["Topic"]
    # Rellenar años vacíos con 0 para que la línea evergreen muestre si se cayó realmente a cero
    años = pd.DataFrame({"post_year": range(2012, agg["post_year"].max() + 1)})
    sub = años.merge(agg[agg["topic"] == t_id], on="post_year", how="left").fillna({"intensidad_relativa": 0, "topic": t_id})
    sub = sub.sort_values("post_year")
    label = f"T{t_id}: {row['terminos_historicos'][:25]}..."
    
    fig_contrast.add_trace(
        go.Scatter(
            x=sub["post_year"], y=sub["intensidad_relativa"],
            mode="lines+markers", name=label,
            line=dict(width=3, color=colores_evergreen[idx % len(colores_evergreen)]),
            hovertemplate=f"<b>{label}</b><br>Año: %{{x}}<br>Intensidad: %{{y:.0f}}% de su pico<extra></extra>"
        ), row=1, col=2
    )

fig_contrast.update_layout(
    title_text="<b>Comportamiento de la Agenda: Relativo al Pico Histórico de cada Tema</b>",
    height=550, width=1100,
    plot_bgcolor="rgba(245,245,245,0.6)",
    legend=dict(orientation="h", yanchor="bottom", y=-0.35, xanchor="center", x=0.5),
    yaxis=dict(title="% de Intensidad (100% = Año de mayor cobertura)"),
    xaxis=dict(dtick=2), xaxis2=dict(dtick=2)
)

fig_contrast.write_html(OUTPUT_DIR / "03B_curvas_intensidad_corregidas.html")
fig_contrast.show()